# Random forest and logistic regression: complete comparison

Start with the four files supplied together in this delivery:

- **RF_logistic_results.ipynb** — this Colab runner.
- **wallet_full_analysis.zip** — prepared real data and saved models.
- **wallet_RFimplementation.zip** — updated component notebooks, helper code and summary renderer.
- **RF_logistic_resultssummary.html** — the latest edited summary from the completed run.

Upload this notebook through Colab's File → Upload notebook menu, then run the code cells in order. When prompted, upload the **two ZIP files, keeping them zipped**. You do not need to upload the HTML to run the models.

Use a standard Python 3 CPU runtime. Step 4 controls whether to retrain all models or reuse the supplied saved models. ROC and calibration each show all four model curves; the smaller-feature detail and middle precision–recall panel are omitted. The regenerated summary includes the RF tuning search and selected settings.

This uses the original prepared features and chronological event-purged splits. The test period was previously inspected, so results remain exploratory. No database password or raw daily trade upload is required for this prepared-data workflow.

## 1. Upload the two ZIP files

Run the cell and choose **both ZIP files**. Wait for both uploads to finish. If you upload the wrong files, the next cell will explain what is missing.

In [ ]:
from google.colab import files
from pathlib import Path

UPLOAD_DIR = Path('/content/wallet_uploads')
UPLOAD_DIR.mkdir(exist_ok=True)
# Colab writes uploads into the current directory.
import os
os.chdir(UPLOAD_DIR)
uploaded = files.upload()
print('Uploaded:', list(uploaded))
del uploaded  # Release the in-memory copy after the files are saved.

## 2. Extract the data and apply the updated code

This keeps the full package's data-preparation notebook and replaces only the updated modeling, importance and summary files. You do not need to copy anything manually or upload the seven raw trade files.

In [ ]:
from pathlib import Path
import zipfile
import shutil

CONTENT = Path('/content')
UPLOAD_DIR = CONTENT / 'wallet_uploads'
identified = {}
for path in sorted(UPLOAD_DIR.glob('*.zip')):
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        if 'wallet_full_analysis/02_notebooks/01_rf_data_preparation.ipynb' in names:
            identified['data'] = path
        if 'wallet_random_forest/02_notebooks/feature_importance.py' in names:
            identified['updated_code'] = path
if set(identified) != {'data', 'updated_code'}:
    raise FileNotFoundError('Upload BOTH wallet_full_analysis.zip and wallet_RFimplementation.zip, keeping them zipped.')

PROJECT_ROOT = CONTENT / 'wallet_full_analysis'
if (PROJECT_ROOT / '.colab_RFimplementation_v2_setup_complete').exists():
    print('Files were already extracted in this session; preserving existing runs.')
else:
    for path in identified.values():
        with zipfile.ZipFile(path) as archive:
            # Validate member paths before extraction.
            for name in archive.namelist():
                (CONTENT / name).resolve().relative_to(CONTENT.resolve())
            archive.extractall(CONTENT)
    updated = CONTENT / 'wallet_random_forest' / '02_notebooks'
    target = PROJECT_ROOT / '02_notebooks'
    for name in ['rf_common.py', 'feature_importance.py', '02_rf_models.ipynb',
                 '03_rf_validation.ipynb', '04_feature_importance_only.ipynb',
                 'comparison_summary.py', 'RF_logistic_resultssummary.template.html']:
        shutil.copyfile(updated / name, target / name)
    (PROJECT_ROOT / '.colab_RFimplementation_v2_setup_complete').write_text('Updated importance code applied.\n')
print('Project ready:', PROJECT_ROOT)
print('Prepared data:', (PROJECT_ROOT / '04_results/03_history_validation').is_dir())

## 3. Install the required Python packages

Run this cell once. If Colab requests a restart, choose **Runtime → Restart session**, then continue at step 4. Uploaded/extracted files remain in the same VM after a Python-session restart. You do not need to upload again unless Colab assigns a new VM.

The saved models were created with scikit-learn 1.8.0, which is explicitly installed here.

In [ ]:
%pip -q install -r /content/wallet_full_analysis/02_notebooks/requirements.txt "scikit-learn==1.8.0"

## 4. Choose whether to retrain

Leave `REFIT_MODELS = True` to reproduce the full modeling analysis: prepare cohorts, tune and fit all ten pipelines, and evaluate the test set.

Set it to `False` if you only want to regenerate the new feature-importance comparison using the provided fitted models. That route skips fitting and new test scoring; it uses the existing test predictions/labels to regenerate plots and the summary.

The runner captures the executed source notebooks with their real cell outputs. It writes all new fitting outputs into a new run folder.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import importlib.metadata
import nbformat
import numpy
import pandas
import sklearn

for package, module in [('numpy', numpy), ('pandas', pandas), ('scikit-learn', sklearn)]:
    if importlib.metadata.version(package) != module.__version__:
        raise RuntimeError('Package versions changed while Python was running. Choose Runtime > Restart session, then resume at step 4.')
if sklearn.__version__ != '1.8.0':
    raise RuntimeError('Run the package installation step, restart the session, then resume here.')

REFIT_MODELS = True  # Change to False for saved-model importance only.

PROJECT_ROOT = Path('/content/wallet_full_analysis')
NOTEBOOK_DIR = PROJECT_ROOT / '02_notebooks'
OUTPUT_BASE = PROJECT_ROOT / '04_results/random_forest'
RUN_TAG = datetime.now(timezone.utc).strftime('colab_%Y%m%dT%H%M%S_%fZ')
EXECUTED_DIR = PROJECT_ROOT / 'executed_colab_notebooks' / RUN_TAG
EXECUTED_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# Explicit paths replace paths saved in the earlier runtime.
os.environ['WALLET_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['WALLET_RF_OUTPUT_BASE'] = str(OUTPUT_BASE)
os.environ['WALLET_VALIDATION_BUNDLE'] = str(PROJECT_ROOT / '04_results/03_history_validation')
os.environ['WALLET_CANDIDATE_PATH'] = str(PROJECT_ROOT / '04_results/01_data_preparation/wallet_market_candidates_full_week.parquet')
os.environ.pop('WALLET_RF_SMOKE', None)
if REFIT_MODELS:
    os.environ.pop('WALLET_RF_RUN_DIR', None)
    os.environ['WALLET_RF_RUN_NAME'] = RUN_TAG
else:
    os.environ['WALLET_RF_RUN_DIR'] = str(OUTPUT_BASE / 'full_data_run')

print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('Mode:', 'retrain models and evaluate' if REFIT_MODELS else 'importance from saved models only')

def run_notebook(filename):
    """Execute source cells sequentially in Colab and save actual outputs."""
    from IPython.utils.capture import capture_output
    nb = nbformat.read(NOTEBOOK_DIR / filename, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == 'code':
            cell.execution_count, cell.outputs = None, []
    # A notebook receives its own namespace while retaining Colab's live kernel.
    namespace = {'__name__': '__wallet_colab_notebook__'}
    count = 0
    print('Running:', filename, flush=True)
    for index, cell in enumerate(nb.cells):
        if cell.cell_type != 'code':
            continue
        count += 1
        cell.execution_count = count
        cell.outputs = []
        print(f'  Code cell {count} ...', flush=True)
        # The code has no notebook magics and executes as ordinary Python.
        # Replay real captured outputs into Colab and retain them in the .ipynb.
        try:
            with capture_output() as captured:
                exec(compile(cell.source, f'{filename}:cell{index}', 'exec'), namespace)
        except Exception as exc:
            cell.outputs = [nbformat.v4.new_output('error', ename=type(exc).__name__,
                            evalue=str(exc), traceback=[])]
            nbformat.write(nb, EXECUTED_DIR / filename)
            raise
        if captured.stdout:
            cell.outputs.append(nbformat.v4.new_output('stream', name='stdout', text=captured.stdout))
        if captured.stderr:
            cell.outputs.append(nbformat.v4.new_output('stream', name='stderr', text=captured.stderr))
        for output in captured.outputs:
            cell.outputs.append(nbformat.v4.new_output('display_data', data=output.data,
                                                       metadata=output.metadata))
        captured.show()
        nbformat.write(nb, EXECUTED_DIR / filename)
    nbformat.validate(nb)
    print('Finished:', filename, flush=True)
    return namespace

## 5. Prepare the cohorts

For a full rerun, this should report 39,659 training rows, 6,956 validation rows, 52,888 final-refit rows and 26,507 test rows. It uses cached features from the supplied real data.

In [ ]:
if REFIT_MODELS:
    preparation = run_notebook('01_rf_data_preparation.ipynb')
else:
    print('Using the provided saved models; preparation is skipped.')

## 6. Tune and fit the models

This is the longest step. The default is eight RF candidate settings, two forward event-purged folds, 200 trees, and all five feature variants. Wait for the cell to finish before proceeding. The full run fits both RF and the logistic comparator; it does not select features by p-values. ROC and calibration show the four baseline/history model curves, and the summary omits the precision–recall panel and smaller-feature detail.

In [ ]:
if REFIT_MODELS:
    fitted = run_notebook('02_rf_models.ipynb')
else:
    print('Saved-model importance mode: fitting is skipped.')

## 7. Display results and updated importance

Full rerun: evaluates frozen models on the original test rows and displays metrics and plots.

Saved-model-only mode: regenerates plots and summary from existing saved models, test predictions/labels and metric tables, without fitting or selecting settings. Logistic importance is |standardized coefficient|, RF importance is Gini. Missingness indicators remain unscaled in logistic and are listed separately. The three correlated price terms appear individually; grouped validation permutation remains a supporting check.

In [ ]:
if REFIT_MODELS:
    evaluated = run_notebook('03_rf_validation.ipynb')
    RUN_DIR = Path(evaluated['RUN_DIR'])
else:
    importance_only = run_notebook('04_feature_importance_only.ipynb')
    RUN_DIR = Path(importance_only['RUN_DIR'])

print('Your results:', RUN_DIR)
print('Importance report:', RUN_DIR / 'figures/RF_logistic_resultssummary.html')

## 8. Download your results

This ZIP includes your selected run's metrics, predictions, fitted models, importance report and plots, plus copies of the notebooks with their actual outputs. It omits raw trades and the source feature-reconstruction inputs.

Download it before ending the session: files inside a Colab runtime can disappear when its VM is recycled. The notebook saved to Drive does not automatically preserve files under `/content`.

In [ ]:
from google.colab import files
import zipfile

archive_path = Path('/content/wallet_colab_results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=str(Path('results') / path.relative_to(RUN_DIR)))
    for path in sorted(EXECUTED_DIR.glob('*.ipynb')):
        archive.write(path, arcname=str(Path('executed_notebooks') / path.name))
    report = RUN_DIR / 'figures' / 'RF_logistic_resultssummary.html'
    archive.write(report, arcname='RF_logistic_resultssummary.html')
    archive.writestr('README.txt', 'Wallet Colab results. See results/figures/RF_logistic_resultssummary.html and the executed_notebooks/ folder. Test-period comparisons remain exploratory.\n')
print('Download size:', round(archive_path.stat().st_size / 1_000_000, 1), 'MB')
files.download(str(archive_path))

## Reading your outputs

- Higher ROC-AUC and average precision are better; lower log loss and Brier score are better. Accuracy uses a fixed 0.5 threshold.
- The earlier run had logistic baseline accuracy 70.11% / AUC 0.7754; RF baseline 68.90% / 0.7695; RF +5 prior-24h history 70.17% / 0.7730. Small numerical differences across environments are possible.
- Importance is not a p-value, causal effect, or proof of trading skill. Gini scores and logistic coefficients have different units. The target is conditional gross profitability, not verified net returns.
- If notebook 02 says the run already evaluated test outcomes, begin a fresh run by rerunning step 4 and then steps 5–7. Do not overwrite a completed run's model choices.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html) and [official upload/download examples](https://colab.research.google.com/notebooks/io.ipynb).